In [1]:
import json
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from rdkit import rdBase, Chem
from tqdm import tqdm

from src import BERT, BERT4IR
from src import build_run_id
from src import load_exps_from_yaml, set_seed, set_default
from src import morgan_tokenize, smiles_tokenize

rdBase.DisableLog('rdApp.warning')

In [2]:
"""
STEP 1: 指定配置文件 ⚠️
- 在此输入你想要执行推理任务的 YAML 路径。
"""

yaml_file = "configs/finetune/task1_morgan_neutral.yaml"

In [3]:
"""
STEP 2: 环境路径配置
- 获取并切换工作目录，确保相对路径指向项目根目录。
"""

current_dir = Path.cwd()
print(f"当前工作目录：{current_dir}")

# 切换至项目根目录
os.chdir("../../")
root_dir = Path.cwd()
print(f"修改后的根目录：{root_dir}")

# 运行设备
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

当前工作目录：D:\Data\Projects\Pycharm\02_Workspace\bert_ir\plots\code
修改后的根目录：D:\Data\Projects\Pycharm\02_Workspace\bert_ir


In [4]:
"""
STEP 3: 数据加载函数
- 通过 seed 确保测试集的可复现性。
- 动态获取标签维度 (out_dim)，适配不同频率范围的光谱长度。
"""


def get_inference_test_data(args):
    dataset = load_dataset('csv', data_files=args.data.path)

    # 1. 执行与训练完全一致的划分流程
    if args.data.split == "c100":
        test_data = dataset['train'].filter(lambda x: x['n_c'] >= 100)
    else:
        # Step A: 80/20 划分出 训练集 / (验证+测试池)
        train_test_data = dataset['train'].train_test_split(test_size=0.2, seed=args.seed)
        val_test_pool = train_test_data['test']
        # Step B: 50/50 划分池子，得到最终的测试集
        val_test_data = val_test_pool.train_test_split(test_size=0.5, seed=args.seed)
        test_data = val_test_data['test']

    # 2. 转换为 Pandas 方便处理
    df_test = test_data.to_pandas()

    # 3. 动态获取输出维度 (out_dim)
    first_label = json.loads(df_test[args.label_col].iloc[0])
    out_dim = len(first_label)

    return df_test, out_dim

In [5]:
"""
STEP 4: 推理辅助工具定义
- get_atom_counts: 统计分子碳原子数及重原子数。
- build_model_from_args: 适配训练逻辑动态构建 BERT4IR 模型。
"""


def get_atom_counts(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 0, 0
    nc = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == 'C')
    n_heavy = mol.GetNumHeavyAtoms()
    return nc, n_heavy


def build_model_from_args(args, out_dim):
    c_dim = args.charge_dim
    bert = BERT(args.vocab_size, args.model.hid_dim, args.model.n_layer,
                args.model.n_head, args.model.dropout, args.rope)
    model = BERT4IR(bert, out_dim, args.data.label_normed, args.support_charge, args.charge.vocab,
                    args.charge.enc, c_dim, c_dim, plot=True)
    return model

In [ ]:
"""
STEP 5: 核心推理引擎
- 调用数据加载函数获取准确的测试集及其维度。
- 执行模型推理并保存 CSV。
"""


def run_single_inference(args, run_id):
    print(f"\n{'=' * 50}\n推理实验: {run_id}\n{'=' * 50}")

    # 1. 获取测试集与动态维度
    df_test, out_dim = get_inference_test_data(args)

    # 2. 定位权重
    model_weight_path = Path(args.save.model) / args.project.name / f"bert4ir_{run_id}.pth"

    if not model_weight_path.exists():
        print(f"[SKIP] 找不到权重文件: {model_weight_path.name}")
        return

    # 3. 加载资源与模型
    with open(args.encode.vocab, 'rb') as f:
        vocab = pickle.load(f)
    args.vocab_size = len(vocab)
    tokenize_fn = smiles_tokenize if args.encode.scheme == "smiles" else morgan_tokenize

    model = build_model_from_args(args, out_dim)

    checkpoint = torch.load(model_weight_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint.get('bert4ir_state_dict', checkpoint))
    model.to(device).eval()

    # 4. 推理
    results = []
    with torch.no_grad():
        for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"推理中", leave=False):
            smi = row["canonical_smiles"]
            label_tensor = torch.tensor(json.loads(row[args.label_col])).unsqueeze(0).to(device)
            charge_tensor = torch.tensor(int(row["charge"])).unsqueeze(0).to(device)

            tokens = tokenize_fn(smi)

            if args.encode.scheme == "morgan":
                tokens = tokens[1::2]

            tokens_idx = [vocab.get(t, vocab["<unk>"]) for t in tokens]
            token_ids = [vocab["<cls>"]] + tokens_idx + [vocab["<sep>"]]
            input_tensor = torch.tensor(token_ids[:args.model.seq_len]).unsqueeze(0).to(device)

            n_c, n_heavy = get_atom_counts(smi)
            outputs = model(input_tensor, label_tensor, charge_tensor)

            results.append({
                "mid": row['mol_id'],
                "formula": row['formula'],
                "canonical_smiles": smi,
                "n_c": n_c,
                "n_heavy": n_heavy,
                "charge": row['charge'],
                "EMD": outputs[2].item(),
                "normed_y_hat": outputs[0].squeeze().cpu().tolist(),
                "normed_y": outputs[1].squeeze().cpu().tolist(),
            })

    # 5. 保存结果
    output_path = Path(f"save/results/test_from_bert/{args.project.name}/{run_id}.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(results).to_csv(output_path, index=False)
    print(f"[DONE] 样本数: {len(df_test)} | 维度: {out_dim} | 平均 EMD: {np.mean([r['EMD'] for r in results]):.4f}")

In [ ]:
"""
STEP 6: 自动化任务分发
- 加载 YAML 配置，自动循环所有参数组合并触发推理引擎。
- 严格拦截：若配置中包含 kfold 参数，则跳过并提示使用专用脚本。
"""

all_exps = load_exps_from_yaml(yaml_file)

for exp_args in all_exps:
    # 1. 拦截包含 kfold 定义的任务
    kfold_val = getattr(exp_args.data, 'kfold', None)

    if kfold_val is not None:
        print(f"\n[?] 拦截实验: {exp_args.run_id}")
        print(f"[!]本脚本仅支持单折(Single Fold)复现，请使用 [generate_results_kfold.ipynb] 文件进行推理。")
        continue

    # 2. 基础环境初始化
    set_seed(getattr(exp_args, "seed", 42))
    exp_args = set_default(exp_args)

    # 3. 展开电荷参数循环
    charge_dim_list = exp_args.charge.onehot_repeat if exp_args.charge.enc == "onehot" else exp_args.charge.emb_dim

    for c_val in charge_dim_list:
        exp_args.charge_dim = c_val

        # 4. 展开数据规模循环
        for scale in exp_args.data.scales:
            # 构建对应的 run_id（不含 K-Fold 逻辑）
            current_run_id = build_run_id(exp_args, scale)
            # 执行推理
            run_single_inference(exp_args, current_run_id)